In [ ]:
import pandas as pd
import numpy as np

In [ ]:
trips = pd.read_csv('../../data/data/processed/trips.csv')
weather = pd.read_csv('../../data/data/processed/weather.csv')
stations = pd.read_csv('../../data/data/processed/stations.csv')

In [ ]:
demand = (trips.groupby(['start_station', 'st_year',
                         'st_month', 'st_day', 'st_hour'])
          .size().reset_index(name='trip_count'))

In [ ]:
demand.tail(20)

In [ ]:
demand['timestamp'] = pd.to_datetime(
    dict(
        year = demand['st_year'],
        month = demand['st_month'],
        day = demand['st_day'],
        hour = demand['st_hour'],
    )
)

In [ ]:
demand = demand.merge(
    stations,
    left_on='start_station',
    right_on='id',
    how = 'left'
)

In [ ]:
demand["date"] = demand["timestamp"].dt.date

In [ ]:
demand.tail()

In [ ]:
weather["date"] = pd.to_datetime(weather["date"]).dt.date

In [ ]:
demand = demand.merge(
    weather,
    on="date",
    how="left"
)

In [ ]:
demand.head()

In [ ]:
demand["hour_sin"] = np.sin(
    2 * np.pi * demand["st_hour"] / 24
)

demand["hour_cos"] = np.cos(
    2 * np.pi * demand["st_hour"] / 24
)

In [ ]:
demand["day_of_week"] = demand["timestamp"].dt.dayofweek

demand["is_weekend"] = (
    demand["day_of_week"] >= 5
).astype(int)

In [ ]:
demand = demand.sort_values(
    ["start_station", "timestamp"]
)

In [ ]:
demand["lag_1hr"] = (
    demand.groupby("start_station")
          ["trip_count"]
          .shift(1)
)

In [ ]:
demand["lag_24hr"] = (
    demand.groupby("start_station")
          ["trip_count"]
          .shift(24)
)

In [ ]:
demand["lag_24hr"] = (
    demand.groupby("start_station")
          ["trip_count"]
          .shift(168)
)

In [ ]:
all_hours = pd.date_range(
    demand["timestamp"].min(),
    demand["timestamp"].max(),
    freq="h"
)

stations_list = trips["start_station"].unique()

In [ ]:
stations_list

In [ ]:
demand

In [ ]:
full_grid = pd.MultiIndex.from_product(
    [stations_list, all_hours],
    names=["start_station", "timestamp"]
).to_frame(index=False)

In [ ]:
fdemand = full_grid.merge(
    demand[["start_station", "timestamp", "trip_count"]],
    on=["start_station", "timestamp"],
    how="left"
)

In [ ]:
fdemand["trip_count"] = (
    fdemand["trip_count"]
    .fillna(0)
    .astype(int)
)

In [ ]:
fdemand["st_year"] = fdemand["timestamp"].dt.year
fdemand["st_month"] = fdemand["timestamp"].dt.month
fdemand["st_day"] = fdemand["timestamp"].dt.day
fdemand["st_hour"] = fdemand["timestamp"].dt.hour

fdemand["day_of_week"] = (
    fdemand["timestamp"].dt.dayofweek
)

fdemand["is_weekend"] = (
    fdemand["day_of_week"] >= 5
).astype(int)

In [ ]:
station = fdemand["start_station"].iloc[0]

(
    fdemand[fdemand["start_station"] == station]
    .sort_values("timestamp")
    .head(30)
)

In [ ]:
print(fdemand.shape)
print(demand.shape)

In [ ]:
fdemand["lag_1hr"] = (
    fdemand.groupby("start_station")
               ["trip_count"]
               .shift(1)
)

In [ ]:
fdemand["lag_24hr"] = (
    fdemand.groupby("start_station")
               ["trip_count"]
               .shift(24)
)

In [ ]:
fdemand["lag_168hr"] = (
    fdemand.groupby("start_station")
               ["trip_count"]
               .shift(168)
)

In [ ]:
fdemand["date"] = fdemand["timestamp"].dt.date
weather["date"] = pd.to_datetime(weather["date"]).dt.date

In [ ]:
fdemand = fdemand.merge(
    weather,
    on="date",
    how="left"
)

In [ ]:
fdemand["hour_sin"] = np.sin(
    2 * np.pi * fdemand["st_hour"] / 24
)

fdemand["hour_cos"] = np.cos(
    2 * np.pi * fdemand["st_hour"] / 24
)

In [ ]:
fdemand["month_sin"] = np.sin(
    2*np.pi*fdemand["st_month"]/12
)

fdemand["month_cos"] = np.cos(
    2*np.pi*fdemand["st_month"]/12
)

In [ ]:
fdemand.head()

In [ ]:
fdemand = fdemand.drop('date', axis=1)

In [ ]:
fdemand.dtypes

In [ ]:
fdemand['st_year'].value_counts()

In [ ]:
fdemand[
    ["lag_1hr","lag_24hr","lag_168hr"]
].isna().sum()

In [ ]:
fdemand = fdemand.merge(
    stations,
    left_on='start_station',
    right_on='id',
    how = 'left'
)

In [ ]:
fdemand.dtypes